# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [2]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [3]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [4]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [5]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [6]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [7]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

In [30]:
# Print the column names and their types
citations.printSchema()
patents.printSchema()

root
 |-- CITING: integer (nullable = true)
 |-- CITED: integer (nullable = true)

root
 |-- PATENT: integer (nullable = true)
 |-- GYEAR: integer (nullable = true)
 |-- GDATE: integer (nullable = true)
 |-- APPYEAR: integer (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- POSTATE: string (nullable = true)
 |-- ASSIGNEE: integer (nullable = true)
 |-- ASSCODE: integer (nullable = true)
 |-- CLAIMS: integer (nullable = true)
 |-- NCLASS: integer (nullable = true)
 |-- CAT: integer (nullable = true)
 |-- SUBCAT: integer (nullable = true)
 |-- CMADE: integer (nullable = true)
 |-- CRECEIVE: integer (nullable = true)
 |-- RATIOCIT: double (nullable = true)
 |-- GENERAL: double (nullable = true)
 |-- ORIGINAL: double (nullable = true)
 |-- FWDAPLAG: double (nullable = true)
 |-- BCKGTLAG: double (nullable = true)
 |-- SELFCTUB: double (nullable = true)
 |-- SELFCTLB: double (nullable = true)
 |-- SECDUPBD: double (nullable = true)
 |-- SECDLWBD: double (nullable = true)



In [9]:
print("citation rows:", citations.count())
print("patent rows:  ", patents.count())

citation rows: 16522438
patent rows:   2923922


In [33]:
# Build a lookup table: patent number -> its state.
patentState = (
    patents
    .select(col("PATENT"), col("POSTATE"))   # 23 cols -> 2
    .filter(col("POSTATE").isNotNull())      # a null state can't be a same-state match
)
# cache() keep this in memory after computing it
patentState.cache()
patentState.show(5)

+-------+-------+
| PATENT|POSTATE|
+-------+-------+
|3070802|     TX|
|3070803|     IL|
|3070804|     OH|
|3070805|     CA|
|3070806|     PA|
+-------+-------+
only showing top 5 rows



In [34]:
# Join 1: look up what state each CITED patent came from.
citedState = (
    patentState.withColumnRenamed("PATENT", "CITED")
    .withColumnRenamed("POSTATE", "CITED_STATE")
)

withCited = citations.join(citedState, on="CITED", how="left")
withCited.show(5)

+-------+-------+-----------+
|  CITED| CITING|CITED_STATE|
+-------+-------+-----------+
|1515701|3858242|       NULL|
|3634889|3858241|         OH|
| 956203|3858241|       NULL|
|1324234|3858241|       NULL|
|3398406|3858241|         FL|
+-------+-------+-----------+
only showing top 5 rows



In [35]:
# Join 2: do the same for the CITING patent, so each row now has both states.
citingState = (
    patentState
    .withColumnRenamed("PATENT", "CITING")
    .withColumnRenamed("POSTATE", "CITING_STATE")
)

bothStates = (
    withCited.join(citingState, on="CITING", how="left")
    .select("CITED", "CITED_STATE", "CITING", "CITING_STATE")
)

bothStates.cache()
bothStates.show(10)

+-------+-----------+-------+------------+
|  CITED|CITED_STATE| CITING|CITING_STATE|
+-------+-----------+-------+------------+
|1331793|       NULL|3858258|          CA|
|1540798|       NULL|3858258|          CA|
| 924225|       NULL|3858527|        NULL|
|3638586|         CA|3858527|        NULL|
|2444326|       NULL|3858527|        NULL|
|3699902|         OH|3858527|        NULL|
|2705120|       NULL|3858527|        NULL|
|2967080|       NULL|3858527|        NULL|
|3602157|         TX|3858527|        NULL|
| 957631|       NULL|3858560|          IN|
+-------+-----------+-------+------------+
only showing top 10 rows



In [37]:
# Keep only same-state pairs, then count them per citing patent.
sameState = (
    bothStates.filter(col("CITED_STATE").isNotNull() &
                              col("CITING_STATE").isNotNull() &
                              (col("CITED_STATE") == col("CITING_STATE")))
)

sameStateCounts = (
    sameState
    .groupBy("CITING")
    .agg(count("*").alias("SAME_STATE"))
    .withColumnRenamed("CITING", "PATENT")
)

sameStateCounts.cache()
sameStateCounts.show(5)

+-------+----------+
| PATENT|SAME_STATE|
+-------+----------+
|3859627|         1|
|3860191|         1|
|3861180|         1|
|3861473|         2|
|3862577|         1|
+-------+----------+
only showing top 5 rows



In [38]:
# Add the new count as an extra column on the full patent table.
patent_same_state_count = patents.join(sameStateCounts, on="PATENT", how="left")
patent_same_state_count.printSchema()

root
 |-- PATENT: integer (nullable = true)
 |-- GYEAR: integer (nullable = true)
 |-- GDATE: integer (nullable = true)
 |-- APPYEAR: integer (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- POSTATE: string (nullable = true)
 |-- ASSIGNEE: integer (nullable = true)
 |-- ASSCODE: integer (nullable = true)
 |-- CLAIMS: integer (nullable = true)
 |-- NCLASS: integer (nullable = true)
 |-- CAT: integer (nullable = true)
 |-- SUBCAT: integer (nullable = true)
 |-- CMADE: integer (nullable = true)
 |-- CRECEIVE: integer (nullable = true)
 |-- RATIOCIT: double (nullable = true)
 |-- GENERAL: double (nullable = true)
 |-- ORIGINAL: double (nullable = true)
 |-- FWDAPLAG: double (nullable = true)
 |-- BCKGTLAG: double (nullable = true)
 |-- SELFCTUB: double (nullable = true)
 |-- SELFCTLB: double (nullable = true)
 |-- SECDUPBD: double (nullable = true)
 |-- SECDLWBD: double (nullable = true)
 |-- SAME_STATE: long (nullable = true)



In [39]:
# The answer: drop the nulls, sort biggest first, show the top 10.
final_output = patent_same_state_count.filter(col("SAME_STATE").isNotNull()) \
                 .orderBy(col("SAME_STATE").desc(), col("PATENT").asc())

final_output.show(10)

# compact form, directly comparable with the RDD notebook's output
print("PATENT     POSTATE  SAME_STATE")
for r in final_output.select("PATENT", "POSTATE", "SAME_STATE").take(10):
    print("%-10d %-8s %d" % (r["PATENT"], r["POSTATE"], r["SAME_STATE"]))

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|SAME_STATE|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|5959466| 1999|14515|   1997|     US|     CA|    5310|      2|  NULL|   326|  4|    46|  159|       0|     1.0|   NULL|  0.6186|    NULL|  4.8868|  0.0455|   0.044|    NULL|    NULL|       125|
|5983822| 1999|14564|   1998|     US|     TX|  569900|      2|  NULL|   114|  5|    55|  200|       0|   0.995|   NULL|  0.7201|    NULL|   12.45|     0.0|     0.0|    NULL|    NULL|       103|
|6008204| 1999|14606|   1998| 

In [40]:
# Shut down Spark.
spark.stop()